In [1]:
# 对称加密例子
from des import DesKey
symKey = DesKey(b"symmetry keycode")                  # for DES

In [2]:
msg = b'this is a secret message!'                    # 明码消息

In [3]:
encryMsg = symKey.encrypt(msg, padding=True)         # 加密消息

In [4]:
encryMsg                                             # 显示密文

b"EgIK\xf6P'\xcflBH4{.>(\x05\xd50\xc0\xd0~\xa2r\xf7\x80\x8a\xbb\xceZu\xcb"

In [5]:
print(symKey.decrypt(encryMsg, padding=True))       # 解密并显示

b'this is a secret message!'


In [6]:
# 非对称加密例子
import rsa

In [7]:
rsa

<module 'rsa' from 'C:\\Users\\sunlu\\anaconda3\\envs\\network\\lib\\site-packages\\rsa\\__init__.py'>

In [8]:
# 创建A、B的公私密钥对
pkA, skA = rsa.newkeys(1024)
pkB, skB = rsa.newkeys(1024)

In [9]:
# 显示密钥对
print('A的公钥是[{}] \n A的私钥是[{}] \n B的公钥是[{}] \n B的私钥是[{}].'.format(pkA, skA, pkB, skB))

A的公钥是[PublicKey(115677343464974655686528140232400147911432286210103827125905437072828984334047389822474898113569945314904966590170206377628099811750886566262482711206046076204817129877128335697625827015813673320243272425064958014029361975444574103226909408355726821462823807274150569621665586971364188646877779887793325207021, 65537)] 
 A的私钥是[PrivateKey(115677343464974655686528140232400147911432286210103827125905437072828984334047389822474898113569945314904966590170206377628099811750886566262482711206046076204817129877128335697625827015813673320243272425064958014029361975444574103226909408355726821462823807274150569621665586971364188646877779887793325207021, 65537, 13866384031323388239824298787947809054308437073204078091781941706885339593333175068211282170075003286599835475111420133705332134841615650160337881344492196184608414814124433833155950240699472841661390237379201901479806060596509192757199451745803632904717251643238848747567320612111666404117133837549683218945, 572251416641184268

In [10]:
# 模拟A将原始消息发送给B，需要用B的公钥加密
raw_messageAtoB = '我的密码是0123456789！！！' # 原始消息
encryptMsgAtoB = rsa.encrypt(raw_messageAtoB.encode(), pkB)

In [11]:
# 模拟中间人嗅探数据包后试图解密消息
# 中间人理论上只有B的公钥
# 1. 直接显示
print(encryptMsgAtoB)

b'\x14\xfa\x91\t\x0e\xb3f\xbb\x04\xd6\x0c\x94%4L\xc1\xf2\xeb\xbd#H\x93\xe8\xac\x88\xd8(<Yv\xdb\xbf!t\r\x1a3\x9f\xb5YO\n\xd9\xf2~\xc9\t\xa6\x87@o\x00\x7ffh\xf5*Z\x963\xd3adM\x1f\xe4\x0e\xbb\xbf\x99\x1e\xd3\x80a\xbc\x04\x01\xc5\x06\x80$P|\xf9\xf4cKL\x8e6\xe3\xc1\x8e\xff\x91;*S=\xcf\xf1\x03\xb1\xef=\xe2\xebg[n\x1eeM\x1c\xcbK\xb4\xda\x0fi3\xca\xban4n\x1e\xc0'


In [12]:
# 2. 用B的公钥解密
print(rsa.decrypt(encryptMsgAtoB, pkB).decode())

AttributeError: 'PublicKey' object has no attribute 'blinded_decrypt'

In [13]:
# 3. 用自己（如A）的私钥解密
print(rsa.decrypt(encryptMsgAtoB, skA).decode())

DecryptionError: Decryption failed

In [14]:
# 4. B用自己的私钥解密
print(rsa.decrypt(encryptMsgAtoB, skB).decode())

我的密码是0123456789！！！


In [15]:
# 模拟B将原始消息发送给A，需要用B的私钥签名以防消息被篡改
raw_messageBtoA = '下周开始计算机网络停课！！！' # 原始消息
signBtoA = rsa.sign(raw_messageBtoA.encode(), skB, 'SHA-512')  # 用SHA-512算法生成原始消息的签名
# 将原始消息与签名合并，并一起发送给A
combMsgBtoA = {'msg':raw_messageBtoA, 'sign':signBtoA}

In [16]:
# 模拟中间人C截获消息并试图篡改消息后发送给A
# 1.首先截获并展示原始消息
print(combMsgBtoA)

{'msg': '下周开始计算机网络停课！！！', 'sign': b"\t\xe6\xfb\xe4\xd6.\xc9\xc7\xef\xf7\x153\xcb\x82\xb74#\x8b\xc4`\x04e\xda!\x0c\x9a\xab\xd2\x99\xd2\xe3\x8a[Hc\xa3\xbae\xe77\x9d\xc3\x0b\xf5\xa5\x89n\xa9l\xfb;\xff\x19o\r\xc1G\x96\xc1Y\xd3\xaf\xdc\x15\xcc\xa4\x05L\xec+\x02\xb5J\xf5\xce*\x11mq$\xc6\xb0\x17\xde}$E\xdc\xdf3\xc5\xdb}\xd2\x07k'5{,F\x87\x17\x81E\x8fW\x11\xf0\x8a\xd0FTFR\xa7\x81\xafuP[\xe6l\xe5\xdcP\x82\xd5"}


In [17]:
# 2.中间人创建自己的密钥对
pkC, skC = rsa.newkeys(1024)

In [18]:
# 3.篡改消息后直接发送给A（不改报文中的签名）
fakeMsgBtoA1 = '下周一晚上计算机网络考试！！！'
combFakeMsgBtoA1 = {'msg':fakeMsgBtoA1, 'sign':signBtoA}

In [19]:
# 4.接收者(A)用B的公钥验证收到的消息combFakeMsgBtoA1
rsa.verify(combFakeMsgBtoA1['msg'].encode(), combFakeMsgBtoA1['sign'], pkB)

VerificationError: Verification failed

In [20]:
# 5.篡改消息后直接发送给A（用自己的私钥签名）
fakeMsgBtoA2 = '下周一晚上计算机网络考试！！！'
combFakeMsgBtoA2 = {'msg':fakeMsgBtoA2, 'sign':rsa.sign(fakeMsgBtoA2.encode(), skC, 'SHA-512')}
print(combFakeMsgBtoA2)

{'msg': '下周一晚上计算机网络考试！！！', 'sign': b"Z\xc8d/\xa0\xceJk\xaf\x94,<!\x8c\xb0\xbbvE[\xdc6{\x80L\xef\xf0\xbd'\xe2\xa6\xda\x83\xac\x1d0\xfa\xe8\xa6:\x88k:\xcd\xef\xc9\x17\x84P\x7f\xd2\xb7#y\xadA\xdc\xc3*bp\xb7\xa8\xf6\tR\x9bg\xfc\xd9\xd6/=\x05\xc2\xda\xd0V\xfd;$v_\xc75\x05\x9b&\x87\x02b\x992\x02<\xa1\xa5}\x0e\xbd\xf7\xd1\x92\x06d\xa5\x85\xd4\xd1K\x90|\xd3#p\xf1\x9b\xbb\xe5d\x1c\x15&P*R\x19\xbf\xea"}


In [21]:
# 6.接收者(A)用B的公钥验证收到的消息combFakeMsgBtoA2
rsa.verify(combFakeMsgBtoA2['msg'].encode(), combFakeMsgBtoA2['sign'], pkB)

VerificationError: Verification failed

In [22]:
# 7.接收者(A)用B的公钥验证收到的正确消息combMsgBtoA
rsa.verify(combMsgBtoA['msg'].encode(), combMsgBtoA['sign'], pkB)

'SHA-512'

In [26]:
# 8.报文摘要示例
import hashlib

In [27]:
# 计算报文摘要方法: 对字符串进行MD5加密
def md5_string(text):
    # 创建MD5对象
    md5 = hashlib.md5()
    # 更新要加密的内容，需要编码为bytes
    md5.update(text.encode('utf-8'))
    # 获取十六进制哈希值
    return md5.hexdigest()

In [28]:
# 使用示例
text1 = "Hello, World!"
text2 = "Hello,World!"
text3 = "华南理工大学自动化工程学院，计算机网络与通讯技术，2026-10-20，上午10:00~12：00."
hashed1 = md5_string(text1)
hashed2 = md5_string(text2)
hashed3 = md5_string(text3)
print("原文: {}---->MD5值: {}".format(text1, hashed1))
print("原文: {}----->MD5值: {}".format(text2, hashed2))
print("原文: {}---->MD5值: {}".format(text3, hashed3))

原文: Hello, World!---->MD5值: 65a8e27d8879283831b664bd8b7f0ad4
原文: Hello,World!----->MD5值: 98f97a791ef1457579a5b7e88a495063
原文: 华南理工大学自动化工程学院，计算机网络与通讯技术，2026-10-20，上午10:00~12：00.---->MD5值: be31948b9c6df4cc14a78bc22de7ea91
